## Library Import

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter, deque
from tqdm import tqdm
import random
import copy
from PIL import Image
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import time
from collections import defaultdict
from pprint import pprint


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
org_path="/mnt/Velocity_Vault/Project_Storage/raw_dataset/"

In [3]:
dataset_org_path = "/mnt/Extra/Project_Storage/stereo_ML_dataset/org/"
dataset_path = "/mnt/Extra/Project_Storage/stereo_ML_dataset/"

left_patch_memmap = "left_patch.dat"
right_strip_memmap = "right_strip.dat"
patch_disparity_memmap = "patch_disp.dat"

## Loading Functions

In [4]:
def load_calibration(file_path):
    
    # {
    #     'cam0': array([[2945.377,    0.   , 1284.862],
    #                 [   0.   , 2945.377,  954.52 ],
    #                 [   0.   ,    0.   ,    1.   ]]),
    #     'cam1': array([[2945.377,    0.   , 1455.543],
    #                 [   0.   , 2945.377,  954.52 ],
    #                 [   0.   ,    0.   ,    1.   ]]),
    #     'doffs': 170.681,
    #     'baseline': 178.232,
    #     'width': 2864,
    #     'height': 1924,
    #     'ndisp': 260,
    #     'isint': 0,
    #     'vmin': 32,
    #     'vmax': 224
    # }
        
    calib = {}
    
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
                
            # Split key and value
            if '=' in line:
                key, value = line.split('=', 1)
                key = key.strip()
                value = value.strip()
                
                # Handle matrix values (e.g., cam0=[...])
                if value.startswith('[') and value.endswith(']'):
                    matrix_str = value[1:-1]
                    # Split rows
                    rows = matrix_str.split(';')
                    matrix = []
                    for row in rows:
                        # Convert each row to float values
                        matrix.append([float(x) for x in row.split()])
                    value = np.array(matrix)
                
                # Handle scalar values
                else:
                    try:
                        # Try to convert to float if possible
                        value = float(value)
                        # Convert to int if no decimal part
                        if value.is_integer():
                            value = int(value)
                    except ValueError:
                        pass  # Keep as string if conversion fails
                
                calib[key] = value
                
    return calib

In [5]:

def load_image_to_rgb(image_path):
    """Load an image from path and return as RGB numpy array."""
    img = Image.open(image_path)
    return np.array(img.convert('RGB'))

In [6]:
def rgb_to_mono(img_array):
    """Convert RGB image array to luminance (grayscale) using standard weights."""
    if len(img_array.shape) == 2:
        return img_array  # Already grayscale
    gray = np.dot(img_array[..., :3], [0.33, 0.33, 0.33])
    return gray.astype(np.int_)

In [7]:

def load_pfm_disparity(file_path):
    
    try:
        with open(file_path, 'rb') as f:
            # Read header lines
            header = f.readline().decode('ascii').strip()
            dimensions = f.readline().decode('ascii').strip()
            scale = f.readline().decode('ascii').strip()
            
            # Parse dimensions
            width, height = map(int, dimensions.split())
            
            # Parse scale and determine endianness
            scale_factor = float(scale)
            little_endian = scale_factor < 0
            
            # Read binary data with correct endianness
            if little_endian:
                data = np.fromfile(f, dtype='<f4')  # Little-endian float32
            else:
                data = np.fromfile(f, dtype='>f4')  # Big-endian float32
            
            # Reshape the data
            expected_size = width * height
            if len(data) == expected_size:
                disparity_matrix = data.reshape((height, width))
            elif len(data) == expected_size * 3:
                # Color image - take mean of RGB channels
                disparity_matrix = data.reshape((height, width, 3)).mean(axis=2)
            else:
                raise ValueError(f"Unexpected data size: got {len(data)}, expected {expected_size}")
            
            disparity_matrix = disparity_matrix[::-1]
            
            disparity_matrix = np.where(disparity_matrix==np.inf,-1,disparity_matrix)
            
            return disparity_matrix
            
    except Exception as e:
        print(f"Error reading PFM file: {e}")
        raise

In [8]:
import numpy as np

def crop_array(array, target_shape):
    
    target_height, target_width = target_shape
    
    arr_shape = array.shape
    
    current_height, current_width = arr_shape[0],arr_shape[1]
    
    start_y = (current_height - target_height) // 2
    start_x = (current_width - target_width) // 2
    end_y = start_y + target_height
    end_x = start_x + target_width
    
    cropped_array = array[start_y:end_y, start_x:end_x]
    
    return cropped_array

## PLotting Functions

In [9]:

def display_image_array(img_array):
    
    # print(img_array.shape)

    
    """Display a numpy image array (2D or 3D) without axes."""
    plt.figure()
    if len(img_array.shape) == 3:  # RGB image
        plt.imshow(img_array)
    else:  # Grayscale
        plt.imshow(img_array, cmap='gray')
    plt.axis('off')
    plt.show()

In [10]:


def resize_image_array(image_array, scale_factor):
    # Convert array to PIL Image
    if len(image_array.shape) == 2:
        # Grayscale image
        img = Image.fromarray(image_array)
    elif len(image_array.shape) == 3:
        # RGB/RGBA image
        img = Image.fromarray(image_array.astype('uint8'))
    else:
        raise ValueError("Input array must be 2D (grayscale) or 3D (color)")
    
    # Calculate new dimensions
    width, height = img.size
    new_width = int(width * scale_factor)
    new_height = int(height * scale_factor)
    
    # Resize using Lanczos resampling (high quality)
    resized_img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
    
    # Convert back to numpy array
    resized_array = np.array(resized_img)
    
    # Preserve original dtype for grayscale
    if len(image_array.shape) == 2:
        resized_array = resized_array.astype(image_array.dtype)
    
    return resized_array

In [11]:
def plot_disp_array(data_array,colourbar=False):
    
    # print(data_array.shape)
    
    plt.figure(figsize=(6, 6))
    img = plt.imshow(data_array, cmap='viridis')
    plt.axis('off')
    if colourbar:
        cbar = plt.colorbar(img, fraction=0.046, pad=0.04)
        cbar.set_label('Value Scale', rotation=270, labelpad=15)
    
    plt.tight_layout()
    plt.show()

In [12]:
def plot_viridis_matrix(matrix):
    # Visualize results
    plt.figure(figsize=(12, 5))
    plt.imshow(matrix, cmap='viridis')
    plt.title('Matrix')
    plt.colorbar()

In [13]:
def display_six_arrays_with_cbar(img1, img2, img3, img4, disp1, disp2, 
                                show_colorbar=False, figsize=(15, 10)):
    """
    Display 6 arrays in a 3x2 grid layout with optional colorbars for disparity maps.
    
    Parameters:
    img1, img2, img3, img4: Image arrays (2D or 3D RGB)
    disp1, disp2: Disparity arrays (2D, displayed with viridis colormap)
    show_colorbar: Whether to show colorbars for disparity maps
    figsize: Tuple specifying the figure size (width, height)
    """
    if show_colorbar:
        fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    else:
        fig, axes = plt.subplots(3, 2, figsize=figsize)
    
    # First row: img1 and img2
    if len(img1.shape) == 3:
        axes[0, 0].imshow(img1)
    else:
        axes[0, 0].imshow(img1, cmap='gray')
    axes[0, 0].axis('off')
    
    if len(img2.shape) == 3:
        axes[0, 1].imshow(img2)
    else:
        axes[0, 1].imshow(img2, cmap='gray')
    axes[0, 1].axis('off')
    
    # Second row: img3 and img4
    if len(img3.shape) == 3:
        axes[1, 0].imshow(img3)
    else:
        axes[1, 0].imshow(img3, cmap='gray')
    axes[1, 0].axis('off')
    
    if len(img4.shape) == 3:
        axes[1, 1].imshow(img4)
    else:
        axes[1, 1].imshow(img4, cmap='gray')
    axes[1, 1].axis('off')
    
    # Third row: disp1 and disp2
    im1 = axes[2, 0].imshow(disp1, cmap='viridis')
    axes[2, 0].axis('off')
    if show_colorbar:
        plt.colorbar(im1, ax=axes[2, 0], fraction=0.046, pad=0.04)
    
    im2 = axes[2, 1].imshow(disp2, cmap='viridis')
    axes[2, 1].axis('off')
    if show_colorbar:
        plt.colorbar(im2, ax=axes[2, 1], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()

## Loading Data

In [14]:
import os

def get_subfolders(folder_path):
    
    try:
        # Check if the path exists and is a directory
        if not os.path.exists(folder_path):
            raise FileNotFoundError(f"Folder not found: {folder_path}")
        
        if not os.path.isdir(folder_path):
            raise NotADirectoryError(f"Path is not a directory: {folder_path}")
        
        # Get all items in the directory
        all_items = os.listdir(folder_path)
        
        # Filter only directories
        subfolders = {item:folder_path+item+"/" for item in all_items 
                    if os.path.isdir(os.path.join(folder_path, item))}
        
        
        return subfolders
        
    except Exception as e:
        print(f"Error: {e}")
        return []

# Example usage:
test_folders = get_subfolders(org_path)

# pprint(test_folders)
print(f"Test Files - {len(test_folders)}")

Test Files - 23


In [15]:
import os

import os

def find_png_files(folder_location):
    png_files = []
    
    if not os.path.exists(folder_location):
        return png_files
    
    try:
        for root, dirs, files in os.walk(folder_location):
            for file in files:
                if file.lower().endswith('.png'):
                    file_name_without_ext = os.path.splitext(file)[0]
                    complete_path = os.path.join(root, file)
                    png_files.append((file_name_without_ext, complete_path))
    except:
        return []
    
    return png_files

def separate_im0_im1(tuple_list):
    im0_locations = []
    im1_locations = []
    
    for item in tuple_list:
        name = item[0]
        if name.startswith('im0e'):
            im0_locations.append(item)
        elif name.startswith('im1e'):
            im1_locations.append(item)
    
    # Sort by the number after 'e' and extract only locations
    im0_locations = [item[1] for item in sorted(im0_locations, key=lambda x: int(x[0].split('e')[1]))]
    im1_locations = [item[1] for item in sorted(im1_locations, key=lambda x: int(x[0].split('e')[1]))]
    
    return im0_locations, im1_locations

In [16]:
calib_locations=[]
rgb_left_locations=[]
rgb_right_locations=[]
truth_left_locations=[]
truth_right_locations=[]

for folder in test_folders:
    
    
    calib_locations.append(test_folders[folder]+"calib.txt")
    truth_left_locations.append(test_folders[folder]+"disp0.pfm")
    truth_right_locations.append(test_folders[folder]+"disp1.pfm")
    
    rgb_left_loc = [test_folders[folder]+"im0.png"]
    rgb_right_loc = [test_folders[folder]+"im1.png"]
    
    for i in range(1,5):
    
        png_files = find_png_files(test_folders[folder]+f"ambient/L{i}/")
        
        if len(png_files)==0:
            continue
        
        left_files,right_files = separate_im0_im1(png_files)
        
        rgb_left_loc +=left_files
        rgb_right_loc +=right_files
        
    rgb_left_locations.append(rgb_left_loc)
    rgb_right_locations.append(rgb_right_loc)
    
# pprint(calib_locations)
# pprint(rgb_left_locations)
# pprint(rgb_right_locations)
# pprint(truth_left_locations)
# pprint(truth_right_locations)


    


In [ ]:


def clean_array(arr, max_disp):
    
    result = np.where(arr < 0, 0, arr)
    finite_mask = np.isfinite(result)
    if np.any(finite_mask):
        max_val = np.max(result[finite_mask])
    else:
        max_val = 0  
    
    result = np.where(np.isinf(result), max_val, result)
    
    if max_val > max_disp:  
        result = (result / max_val) * max_disp
    
    result = result.astype(np.int16)
    
    return result

def load_data_from_files(resize_fraction, max_disp):
    resize_factor = 1/resize_fraction
    
    size = len(calib_locations)
    
    calib_files = []
    truth_files = {'org':[],'flip':[]}
    image_files = {'org':[],'flip':[]}
    
    for i in tqdm(range(size)):
        
        try:
        
            calib = load_calibration(calib_locations[i])
            calib_files.append(calib)
            
            truth_l = load_pfm_disparity(truth_left_locations[i])
            truth_r = load_pfm_disparity(truth_right_locations[i])
            
            truth_l = resize_image_array(truth_l,resize_factor)
            truth_l = truth_l//resize_fraction
            truth_r = resize_image_array(truth_r,resize_factor)
            truth_r = truth_r//resize_fraction
            
            truth_l = clean_array(truth_l,max_disp)
            truth_r = clean_array(truth_r,max_disp)
            
            
            truth_org_dict = {"index":i,'left':np.array(truth_l,dtype=np.int16),'right':np.array(truth_r,dtype=np.int16)}
            
            truth_files['org'].append(truth_org_dict)
            
            truth_l_flip = np.fliplr(truth_l)
            truth_r_flip = np.fliplr(truth_r)
            
            truth_flip_dict = {"index":i,'left':np.array(truth_r_flip,dtype=np.int16),'right':np.array(truth_l_flip,dtype=np.int16)}

            truth_files['flip'].append(truth_flip_dict)
            
            org_dict_store = []
            flip_dict_store = []
            
            for image_index in range(len(rgb_left_locations)):
                
                try:
                
                    if image_index >= len(rgb_right_locations[i]):
                        break
                    
                    rgb_l = load_image_to_rgb(rgb_left_locations[i][image_index])
                    rgb_r = load_image_to_rgb(rgb_right_locations[i][image_index])
                    
                    rgb_l = resize_image_array(rgb_l,resize_factor)
                    rgb_r = resize_image_array(rgb_r,resize_factor)
                    
                    gray_l = rgb_to_mono(rgb_l)
                    gray_r = rgb_to_mono(rgb_r)
                    
                    image_org_dict = {"index":i,'left':np.array(gray_l,dtype=np.uint8),'right':np.array(gray_r,dtype=np.uint8)}
                    
                    org_dict_store.append(image_org_dict)
                    
                    gray_l_flip = np.fliplr(gray_l)
                    gray_r_flip = np.fliplr(gray_r)
                    
                    image_flip_dict = {"index":i,'left':np.array(gray_r_flip,dtype=np.uint8),'right':np.array(gray_l_flip,dtype=np.uint8)}
                    
                    flip_dict_store.append(image_flip_dict)
                    
                except Exception as e:
                    print(rgb_left_locations[i][image_index])
                    print(rgb_right_locations[i][image_index])
                    print(e)
                    
            image_files['org'].append(org_dict_store)
            image_files['flip'].append(flip_dict_store)
            
        except Exception as e:
            print(truth_left_locations[i])
            print(truth_right_locations[i])
            print(e)
            
    
            
    return calib_files,image_files,truth_files
        

In [18]:
import pickle

def save_variable(file_location, variable):
    with open(file_location, 'wb') as f:
        pickle.dump(variable, f)

resize_fraction = 1
resize_factor = 1/resize_fraction


maximum_disparity = int((700)*resize_factor)


calib_files,image_files,truth_files = load_data_from_files(resize_fraction, maximum_disparity)

# print(len(calib_files))
# print(len(image_files['org']))
# print(len(image_files['flip']))
# print(len(image_files['org'][2]))
# print(len(image_files['flip'][2]))
# print(len(truth_files['org']))
# print(len(truth_files['flip']))

variable_org_path = "/mnt/Velocity_Vault/Project_Storage/stereo_ML_dataset/raw_images/"

save_variable(variable_org_path+"calib.pkl", calib_files)
save_variable(variable_org_path+"image.pkl", image_files)
save_variable(variable_org_path+"truth.pkl", truth_files)

100%|██████████| 23/23 [05:26<00:00, 14.17s/it]
